# personal_compare – Personalisierungs-Pipeline und Session-Analysen

**Ziel:** Ein auf GazeCapture vortrainiertes Modell (ResNet-18 + MLP-Kopf, Sigmoid) wird pro Person/Session mit wenigen eigenen Webcam-Bildern nachtrainiert (nur der Kopf, *FC only*).

**Sessions (18):** Gen 1 = 01–04 (`base_Code.py`), Gen 2 = 05–09 (`base_Code_rand.py`), Gen 4 = 19–27 (`base_Code_rand2.py`, WAIT/MOVE-Protokoll mit Phasen-Logging).
Gen 3 (Sessions 10–18) wurde verworfen und wird hier **nicht** verwendet; die Ordner bleiben unverändert.

| Abschnitt | Inhalt |
|---|---|
| 1 | Setup |
| 2 | Datensatz, Transformationen, Split 70/15/15 |
| 3 | Training einer Session (Early Stopping auf Validierung, Test am besten Validierungs-Epoch) |
| 4 | Session-Baselines (konstant / zufällig) |
| 5 | Läufe: Original vs. bereinigt (WAIT entfernt), Basismodell `norm_subject` vs. `norm_random` |
| 6 | Gespeicherte Ergebnisse (Logs) – korrigierte Metrik |
| 7 | Vergleich mit Baselines |
| 8 | Original vs. bereinigt (Gen 4) |
| 9 | Ablationen: Brille, Hintergrund, Aufnahmedauer, Betriebssystem |

> **Metrik-Bugfix (Okt. 2026):** Die frühere Funktion `diagonal_errors` nutzte `sqrt(mean_squared_error)/√2` und lieferte um den Faktor √2 zu kleine Fehler.
> Die Session-Baselines (`personal_code.ipynb`) waren dagegen euklidisch korrekt. Alle gespeicherten Modellwerte werden in Abschnitt 6 exakt mit √2 umgerechnet.

> **Begriff:** In den Ergebnistabellen bedeutet `norm_subject`/`norm_random` **welches GazeCapture-Basismodell** geladen wurde.
> Innerhalb einer Session ist die Aufteilung immer zufällig (70/15/15, Seed 42).

## 1. Setup

In [ ]:
import os, sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from metrics import diagonal_error_pct, constant_baseline_pct, random_baseline_pct

SESSIONS_GEN = {"Gen 1": ["01", "02", "03", "04"],
                "Gen 2": ["05", "06", "07", "08", "09"],
                "Gen 4": ["19", "20", "21", "22", "23", "24", "25", "26", "27"]}
SESSIONS = [s for v in SESSIONS_GEN.values() for s in v]
GEN_OF = {s: g for g, v in SESSIONS_GEN.items() for s in v}

# Aufnahmebedingungen (aus dem Session-Protokoll)
META = pd.DataFrame([
    # session, glasses, background/light, duration_s, os
    ("01", "with", "bright", None, "Ubuntu"), ("02", "without", "bright", None, "Ubuntu"),
    ("03", "with", "dark", None, "Ubuntu"), ("04", "with", "bright (library)", None, "Ubuntu"),
    ("05", "with", "bright", None, "Ubuntu"), ("06", "without", "bright", None, "Ubuntu"),
    ("07", "with", "dark", None, "Ubuntu"), ("08", "with", "very dark", None, "Ubuntu"),
    ("09", "without", "very dark", None, "Ubuntu"),
    ("19", "with", "clear", 120, "Ubuntu"), ("20", "with", "clear", 90, "Ubuntu"),
    ("21", "without", "clear", 120, "Ubuntu"), ("22", "without", "clear", 90, "Ubuntu"),
    ("23", "with", "noisy", 120, "Ubuntu"), ("24", "without", "noisy", 120, "Ubuntu"),
    ("25", "with", "noisy", 120, "Windows"), ("26", "without", "noisy", 120, "Windows"),
    ("27", "with", "noisy", 90, "Windows"),
], columns=["session", "glasses", "background", "duration_s", "os"])
META["generation"] = META.session.map(GEN_OF)
META

## 2. Datensatz, Splits und Training → `personal_exp.py`
Die gesamte Personalisierungs-Logik liegt in **`personal_exp.py`** (Projektwurzel), damit Notebook und Kommandozeile dieselbe Implementierung nutzen.

* Basismodell: `models/best_models/best_ResNet_Sigmoid_{base}.path` (GazeCapture), nur der Kopf (`model.fc`) wird trainiert
* AdamW (lr = 1e-4, weight decay = 1e-5), Batch 32, Early Stopping (Patience 3) auf dem **Validierungsfehler**; berichtet wird der **Testfehler am besten Validierungs-Epoch**
* Epoche 0 = Basismodell **ohne** Personalisierung → `epoch0_test_error` zeigt den Gewinn durch die Personalisierung
* Loss: `standard` = `nn.L1Loss`, `sensitive` = positionsgewichteter L1

| Split (`--split`) | Train / Val / Test | misst |
|---|---|---|
| `random` | 70/15/15 zufällige Frames | optimistisch: benachbarte, fast gleiche Frames liegen in Train **und** Test |
| `sequential` | erste 70 % / nächste 15 % / letzte 15 % (zeitlich) | Übertragung auf einen **späteren** Teil der Aufnahme (realistisch) |
| `two_videos` | Train/Val aus Session A (85/15), Test = ganze Session B | Übertragung auf eine **andere Aufnahme** derselben Person |

In [ ]:
import personal_exp as pe
n = 100
for sp in ("random", "sequential"):
    tr, va, te = pe.split_indices(n, sp, seed=42)
    print(f"{sp:10s} train {len(tr)}  val {len(va)}  test {len(te)}  erste Test-Indizes: {te[:5]}")

## 3. Ein einzelner Lauf (Test)
Für viele Läufe besser im Terminal: `python personal_exp.py --sessions all --base norm_subject norm_random`
(jeder fertige Lauf wird sofort in `results/personal/personal_runs.jsonl` gespeichert; ein abgebrochener Lauf wird beim nächsten Start fortgesetzt).

In [ ]:
RUN_SINGLE = False
if RUN_SINGLE:
    r = pe.run_one("02", base="norm_subject", labels="original", split="random")
    print({k: v for k, v in r.items() if k != "history"})

## 4. Session-Baselines
Konstant = Mittelwert der Trainingsziele der Session, ausgewertet auf dem Test-Split derselben Session.

In [ ]:
def session_baselines(session, labels="original", split="random", seed=42):
    df = pe.load_labels(session, labels)
    tr, _, te = pe.split_indices(len(df), split, seed)
    xy = df[["x", "y"]].to_numpy(float)
    return dict(session=session, n=len(df), split=split, constant=constant_baseline_pct(xy[tr], xy[te]),
                random=random_baseline_pct(xy[te]))

if Path("personalization/01/norm_labels.csv").exists():
    display(pd.DataFrame([session_baselines(s, split=sp) for sp in ("random", "sequential") for s in SESSIONS]).round(2))

## 5. Neue Läufe (aus `personal_runs.jsonl`)
Wird gefüllt durch `python personal_exp.py ...` (siehe README, Experimente E1–E3).

In [ ]:
runs = pe.summary()          # Tabelle aller gespeicherten Läufe + Mittelwerte je Konfiguration

## 6. Gespeicherte Ergebnisse (historische Logs)
Quelle: Ausgaben der früheren `personal_compare.ipynb` (Zellen „Final diagonal Test Error“) und der Session-Tabellen.
`*_old` = alte Metrik, die übrigen Spalten = **korrigierte Metrik** (× √2). Konstant-Baseline aus `personal_code.ipynb` (Mittelwert der gesamten Session, euklidisch).

In [ ]:
S2 = np.sqrt(2)
hist = pd.DataFrame({
    "session": SESSIONS,
    # Basismodell norm_subject, Original-Labels: Test-Fehler am besten Val-Epoch (alte Metrik)
    "test_subj_old":  [11.43, 8.52, 12.10, 10.84, 12.78, 10.49, 15.44, 19.50, 18.51,
                       12.48, 16.17, 9.35, 11.80, 15.19, 16.44, 19.14, 13.81, 18.65],
    # Basismodell norm_random (nur Sessions 01-09 geloggt)
    "test_rand_old":  [10.24, 9.13, 15.50, 11.15, 15.58, 11.08, 16.26, 20.17, 14.99] + [np.nan] * 9,
    # bereinigte Labels (WAIT entfernt), nur Gen 4
    "test_clean_old": [np.nan] * 9 + [11.79, 12.30, 9.35, 12.48, 17.07, 16.83, 15.65, 13.54, 18.08],
    "const_baseline": [28.77637, 29.30417, 28.75691, 28.50943, 30.88308, 27.4493, 28.45881, 26.94685, 25.86613,
                       30.79236, 31.82363, 31.62243, 30.57442, 31.80651, 33.09283, 31.25057, 29.8298, 31.52212],
    "random_baseline": [41.05408, 41.00951, 40.73115, 38.45547, 43.16937, 39.71674, 41.85134, 39.97005, 40.90327,
                        42.14856, 41.35664, 41.97698, 43.36335, 42.96989, 44.35355, 42.99306, 41.58859, 43.14927],
})
for c in ("test_subj", "test_rand", "test_clean"):
    hist[c] = hist[c + "_old"] * S2
hist = hist.merge(META, on="session")
hist["reduction_vs_const_%"] = 100 * (1 - hist["test_subj"] / hist["const_baseline"])
hist.round(2)

## 7. Vergleich mit den Baselines

In [ ]:
def ci98(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    m = x.mean(); h = stats.t.ppf(0.99, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return round(m, 2), round(m - h, 2), round(m + h, 2), len(x)

summary = (hist.groupby("generation")[["test_subj", "const_baseline", "random_baseline"]]
           .agg(["mean"]).round(2))
display(summary)
print("Alle 18 Sessions – personalisiert:", ci98(hist.test_subj), "| konstant:", ci98(hist.const_baseline))
print("Gepaarter t-Test personalisiert vs. konstant:", stats.ttest_rel(hist.test_subj, hist.const_baseline))
worse = hist[hist.test_subj >= hist.const_baseline].session.tolist()
print("Sessions ohne Gewinn gegenüber der Konstant-Baseline:", worse)

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(hist)); w = 0.4
ax.bar(x - w/2, hist.const_baseline, w, color="#9e9e9e", label="constant baseline")
ax.bar(x + w/2, hist.test_subj, w, color="#1f77b4", label="personalized (FC only)")
ax.set_xticks(x, hist.session); ax.set_ylabel("Diagonal error (%)"); ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("results/figures/sessions_vs_baseline.png", dpi=200); plt.show()

### Basismodell `norm_subject` vs. `norm_random` (Sessions 01–09)

In [ ]:
d = hist.dropna(subset=["test_rand"])
print("norm_subject:", ci98(d.test_subj), "| norm_random:", ci98(d.test_rand))
print(stats.ttest_rel(d.test_subj, d.test_rand))

## 8. Wiederholungslauf auf identischen Daten (Gen 4)

> **Korrektur (Okt. 2026):** Die früheren Dateien `norm_labels_clean.csv` waren byte-identisch mit `norm_labels.csv` (gleiche Zeilenzahl im Log: 574/540/720). Die Spalte `test_clean` ist daher **kein** Effekt der Bereinigung, sondern ein zweiter Lauf auf denselben Daten → sie zeigt die Lauf-zu-Lauf-Streuung (Ø |Δ| = 1,86 pp, max. 5,47 pp).
>
> Korrekte bereinigte Dateien: `python session_design.py --write-clean`; neuer Vergleich: Experiment E1b (`python personal_exp.py --sessions gen4 --labels clean`).

In [ ]:
g4 = hist[hist.generation == "Gen 4"].copy()
g4["delta_clean"] = g4.test_clean - g4.test_subj
display(g4[["session", "glasses", "background", "duration_s", "os", "test_subj", "test_clean", "delta_clean"]].round(2))
print("Mittel original:", ci98(g4.test_subj), "| bereinigt:", ci98(g4.test_clean))
print(stats.ttest_rel(g4.test_clean, g4.test_subj))

## 9. Ablationen (nur Gen 4, gleiche Aufnahme-Software)
Die Faktoren sind **nicht** vollständig balanciert (z. B. alle Windows-Sessions haben einen unruhigen Hintergrund). Die Mittelwerte sind daher beschreibend; bei n = 2–6 pro Gruppe sind keine Signifikanzaussagen möglich.

In [ ]:
for factor in ("glasses", "background", "duration_s", "os"):
    t = g4.groupby(factor)[["test_subj", "test_clean"]].agg(["mean", "count"]).round(2)
    print(f"\n=== {factor} ==="); display(t)

# Gepaarte Vergleiche, bei denen nur EIN Faktor wechselt:
pairs = {"Brille (mit - ohne), clear 120 s Ubuntu": ("19", "21"),
         "Brille (mit - ohne), clear 90 s Ubuntu": ("20", "22"),
         "Brille (mit - ohne), noisy 120 s Ubuntu": ("23", "24"),
         "Brille (mit - ohne), noisy 120 s Windows": ("25", "26"),
         "Dauer (90 - 120 s), mit Brille, clear": ("20", "19"),
         "Dauer (90 - 120 s), ohne Brille, clear": ("22", "21"),
         "Hintergrund (noisy - clear), mit Brille, 120 s": ("23", "19"),
         "Hintergrund (noisy - clear), ohne Brille, 120 s": ("24", "21"),
         "OS (Windows - Ubuntu), mit Brille, noisy 120 s": ("25", "23"),
         "OS (Windows - Ubuntu), ohne Brille, noisy 120 s": ("26", "24")}
v = g4.set_index("session")
pd.DataFrame([(k, round(v.loc[a, "test_subj"] - v.loc[b, "test_subj"], 2),
               round(v.loc[a, "test_clean"] - v.loc[b, "test_clean"], 2)) for k, (a, b) in pairs.items()],
             columns=["Vergleich", "Δ original (pp)", "Δ bereinigt (pp)"])